# Init package

In [18]:
%cd /home/FullMouth/code
import os
from pathlib import Path
from glob import glob
import pandas as pd
import numpy as np
import ast
import json
import random
import shutil
from tqdm import tqdm

STR_NA = 'Na'
STR_HALU  = 'Hallu'

from fullmouth_util import (
    FM_label, ROUND_THRES, ENTITY_LS, ENTITY_DICT, OUTPUT_FORMAT, SENT, STOP, write_json, load_json
)
fm_label = FM_label()

src_train_dir = r"/home/FullMouth/data/"
add_stop_token = True

In [19]:
model_root = r'/data_sys/lang_model/4_n20'

ls = [fn for fn in os.listdir(model_root) if 'ep' in fn and 'Qwen' in fn]
ls.sort()
for i, model in enumerate(ls):
    print(i, model)

In [2]:
data_root = r'/home/FullMouth/data/instruct_4_n20'
data_type = '_revisedTrain_withDesc_withEx_withErrFeed'

model_dir = 'Qwen2.5-14B-Instruct_revisedTrain_withDesc_withEx_withErrFeed_ep04r16_medium'
model_name, post_fix = model_dir.split(data_type)

data_dir = os.path.join(data_root, f'{model_name}{data_type}')
print([model_name, post_fix])
print([data_dir, os.path.exists(data_dir)])
assert os.path.exists(data_dir), f'{data_dir} does not exist! Please check the data_dir path.'

gold_sft_dir = os.path.join(data_dir, 'gold_sft_3instructs09')
train_sft_dir = os.path.join(data_dir, f'train_sft_3instructs09')

# Setting

In [5]:
# model_name = model_name #'granite-3.1-8b-instruct'
data_type = 'revisedTrain_withDesc_withEx_withErrFeed'
model_setting = post_fix[1:]

prompt_selected = '3instructs09'
train_dir_name = f'train_sft_{prompt_selected}_{model_setting}'


version = f'4_n20'
root_dir = r'/home/FullMouth/data'

target_model_dir = os.path.join(root_dir, f'instruct_{version}',f'{model_name}_{data_type}' )

selected_prompt_fp = os.path.join(target_model_dir, f'selected_prompt_{prompt_selected}.json')
instruct_dict_selected = load_json(selected_prompt_fp)

train_sft_dir = os.path.join(target_model_dir, train_dir_name)
json_fp_ls = glob( os.path.join(train_sft_dir, '*.json'))
assert len(json_fp_ls) > 0, f"No .json files found in {train_sft_dir}"
print(version, target_model_dir)
print(train_sft_dir, len(json_fp_ls))

In [6]:
output_dir = os.path.join(target_model_dir, f'DPO_data_{model_setting}_{prompt_selected}')
print(output_dir)
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)
Path(output_dir).mkdir(parents=True, exist_ok=True)

# DPO `chosen` vs `rejected`

In [7]:
STR_HALU = '<hallucination>'
STR_MISSING = '<missing>'
STR_NA = '<N/A>'

STR_CHOSEN = 'chosen'
STR_REJECTED = 'rejected'
STR_PROMPT = 'prompt'

##### Combine

In [8]:
from fullmouth_util import (
    PRED, ENTITY_DICT, INSTRUCT_PROMPT, SENT, is_similarity, ENTITY_LS
)

from function_util import (
    get_msg_ls_checkInstruction, get_msg_ls_resultsInstruction
)


def replace_msg_ls_to_user(msg_ls: list[dict]) -> list[dict]:
    """
    Some models require user/assistant only
    """
    for i, msg in enumerate(msg_ls):
        if msg['role'] == 'system':
            msg_ls[i]['role'] = 'user'
    
    # Combine consecutive messages with the same role into one message
    combined_msg_ls = []
    for msg in msg_ls:
        if not combined_msg_ls:
            combined_msg_ls.append(msg)
        else:
            last_msg = combined_msg_ls[-1]
            if msg['role'] == last_msg['role']:
                # Combine the content of the messages
                combined_msg_ls[-1]['content'] += '\n' + msg['content']
            else:
                combined_msg_ls.append(msg)
    
    return combined_msg_ls

def get_json_str(d):
    return json.dumps(d, ensure_ascii=False, sort_keys=True, separators=(",", ":"))

def get_assist_msg(content):
    return {'role': 'assistant', 'content': content}

# Collect DPO data

In [9]:
for json_fp in json_fp_ls:
    json_fn = os.path.basename(json_fp)
    dpo_ls_fp = os.path.join(output_dir, json_fn)

    gold_collect_ls = load_json(os.path.join(src_train_dir, json_fn))
    collect_ls = load_json(json_fp)
    assert len(gold_collect_ls) == len(collect_ls)
    # collect_ls = update_collect_ls(collect_ls, gold_collect_ls)
    dpo_ls = []
    
    for sent_idx, sent_dict in enumerate(collect_ls):
        pred_dict = sent_dict.get(PRED, {})
        gold_dict = gold_collect_ls[sent_idx][ENTITY_DICT]
        if not pred_dict and not gold_dict: continue
        if pred_dict and gold_dict:
            for entity in gold_dict:
                unit_dpo_ls = []
                if entity not in fm_label.gold_entity_ls: continue
                for prompt_idx, instruct_prompt_dict in instruct_dict_selected[entity].items():
                    instruct_prompt = instruct_prompt_dict[INSTRUCT_PROMPT]
                    
                    if entity in pred_dict and isinstance(pred_dict[entity], list):
                        pred2gold_is_similar = all([is_similarity(entity_pred_val, gold_dict[entity]) for entity_pred_val in pred_dict[entity]])
                        gold2Pred_is_similar = all([is_similarity(entity_gold_val, pred_dict[entity]) for entity_gold_val in gold_dict[entity]])
                        # print('pred2gold_is_similar', pred2gold_is_similar, 'gold2Pred_is_similar', gold2Pred_is_similar)
                        if pred2gold_is_similar and gold2Pred_is_similar:
                            continue
                        else:
                            rejected_dict = {SENT: sent_dict[SENT], ENTITY_LS: pred_dict[entity]}
                    else:
                        rejected_dict = {}

                    result_msg_ls = get_msg_ls_resultsInstruction(instruct_prompt, sent_dict[SENT], add_stop_token=add_stop_token)
                    chosen_val   = [get_assist_msg( get_json_str({SENT: sent_dict[SENT], ENTITY_LS: gold_dict[entity]}) )]
                    rejected_val = [get_assist_msg( get_json_str( rejected_dict) )]
                    unit_dpo_ls.append({STR_PROMPT: result_msg_ls, STR_CHOSEN: chosen_val, STR_REJECTED: rejected_val,})
                
                if unit_dpo_ls: dpo_ls.append(unit_dpo_ls)

            # For predicted entity **not** in gold entity, thus block on checkInstruction
            for entity in pred_dict:
                unit_dpo_ls = []
                if entity not in gold_dict and entity in fm_label.gold_entity_ls:
                    for prompt_idx, instruct_prompt_dict in instruct_dict_selected[entity].items():
                        instruct_prompt = instruct_prompt_dict[INSTRUCT_PROMPT]
                        checked_msg_ls = get_msg_ls_checkInstruction(instruct_prompt, sent_dict[SENT], entity)
                        chosen_val = [ get_assist_msg( "No" )]
                        rejected_val = [ get_assist_msg( "Yes" )]
                        unit_dpo_ls.append({STR_PROMPT: checked_msg_ls, STR_CHOSEN: chosen_val, STR_REJECTED: rejected_val,})
                
                if unit_dpo_ls: dpo_ls.append(unit_dpo_ls)
                    
        elif not pred_dict and gold_dict: # model missed all entities
            for entity in gold_dict:
                unit_dpo_ls = []
                if entity not in fm_label.gold_entity_ls: continue

                for prompt_idx, instruct_prompt_dict in instruct_dict_selected[entity].items():
                    instruct_prompt = instruct_prompt_dict[INSTRUCT_PROMPT]
                    checked_msg_ls = get_msg_ls_checkInstruction(instruct_prompt, sent_dict[SENT], entity)
                    chosen_val =  [ get_assist_msg( "Yes" )]
                    rejected_val =  [ get_assist_msg( "No" )]
                    unit_dpo_ls.append({STR_PROMPT: checked_msg_ls, STR_CHOSEN: chosen_val, STR_REJECTED: rejected_val,})

                    result_msg_ls = get_msg_ls_resultsInstruction(instruct_prompt, sent_dict[SENT], add_stop_token=add_stop_token)
                    chosen_val =  [ get_assist_msg( get_json_str({SENT: sent_dict[SENT], ENTITY_LS: gold_dict[entity]}) )]
                    rejected_val = [ get_assist_msg( "{}") ]
                    unit_dpo_ls.append({STR_PROMPT: result_msg_ls, STR_CHOSEN: chosen_val, STR_REJECTED: rejected_val,})
                
                if unit_dpo_ls: dpo_ls.append(unit_dpo_ls)

        elif not gold_dict and pred_dict: # False positive cases
            for entity in pred_dict:
                unit_dpo_ls = []
                if entity not in fm_label.gold_entity_ls: continue
                for prompt_idx, instruct_prompt_dict in instruct_dict_selected[entity].items():
                    instruct_prompt = instruct_prompt_dict[INSTRUCT_PROMPT]
                    
                    checked_msg_ls = get_msg_ls_checkInstruction(instruct_prompt, sent_dict[SENT], entity)
                    chosen_val = [ get_assist_msg( "No" )]
                    rejected_val = [ get_assist_msg( "Yes" )]
                    unit_dpo_ls.append({STR_PROMPT: checked_msg_ls, STR_CHOSEN: chosen_val, STR_REJECTED: rejected_val,})
                    
                    result_msg_ls = get_msg_ls_resultsInstruction(instruct_prompt, sent_dict[SENT], add_stop_token=add_stop_token)
                    chosen_val = [ get_assist_msg( "{}" )]
                    rejected_val = [ get_assist_msg( get_json_str({SENT: sent_dict[SENT], ENTITY_LS: pred_dict[entity]}) )]
                    unit_dpo_ls.append({STR_PROMPT: result_msg_ls, STR_CHOSEN: chosen_val, STR_REJECTED: rejected_val,})
                
                if unit_dpo_ls: dpo_ls.append(unit_dpo_ls)

    if dpo_ls:
        # print('saved')
        # for case_id, case_ls in enumerate(dpo_ls):
        #     for msg_idx, msg_dict in enumerate(case_ls):
        #         dpo_ls[case_id][msg_idx]['prompt'] = replace_msg_ls_to_user(msg_dict['prompt'])
        
        write_json(dpo_ls, dpo_ls_fp)

# Save DPO to jsonl

In [10]:
from sklearn.model_selection import train_test_split

In [11]:
jsonl_fp_train = os.path.join(output_dir, 'dpo_data_train.jsonl')
jsonl_fp_val = os.path.join(output_dir, 'dpo_data_val.jsonl')

json_fp_ls = glob(os.path.join(output_dir, '*.json'))
dpo_ls = [ unit_dpo_ls  for json_fp in json_fp_ls[:] for unit_dpo_ls in load_json(json_fp)  if unit_dpo_ls ]

print(jsonl_fp_train)
print(jsonl_fp_val)
print(len(json_fp_ls), len(dpo_ls))
dpo_train, dpo_val = train_test_split(dpo_ls, test_size=0.2, random_state=42)

print(f"Train set size: {len(dpo_train)} ({len(dpo_train)/len(dpo_ls)*100:.1f}%)")
print(f"Val set size: {len(dpo_val)} ({len(dpo_val)/len(dpo_ls)*100:.1f}%)")
print(f"Total: {len(dpo_train) + len(dpo_val) }")

for jsonl_fp, dpo_data in [ (jsonl_fp_train, dpo_train), (jsonl_fp_val, dpo_val),]:
    count = 0
    with open(jsonl_fp, "w", encoding="utf-8") as f:
        for unit_dpo_ls in dpo_data:
            for unit in unit_dpo_ls:
                count += 1
                f.write(json.dumps(unit, ensure_ascii=False) + "\n")
    print(f"Total units written to {os.path.basename(jsonl_fp)}: {count}")
